# 재사용 이미지 VQA 앙상블 베이스라인

앞의 `VQA_Gemma31B_Reusable_Baseline_v1.ipynb`가 만든 **실제 NPY 확률·ID·정답** 및 모델별 hard CSV를 연결한다. 코드·설정은 모두 내장했다. 여러 모델뿐 아니라 동일 모델의 서로 다른 run도 등록할 수 있다.

- 기본: 동일 validation 검증 → 이미지 그룹별 meta-fit/meta-holdout → 기준 모델 선택 → soft/hard 후보 비교 → 조건부 기준 모델 교체 → 결과 저장.
- 선택: 확률 온도 보정, 학습형 soft 가중치, core3·4/8개 만장일치, Logistic/RF 메타 모델, 수동 검토 기록.
- 기준 모델은 Public 순위로 자동 선택하지 않는다. 기본 `best_on_meta_fit`이며 명시적으로 고정할 수도 있다.
- 1,434/6,714는 고정하지 않는다. 672·1,343·1,434 validation의 교집합을 몰래 만들어 비교하지 않는다.
- hard 답은 투표·메타 모델의 명시적 범주형 feature로만 사용한다. 실제 soft 입력으로 둔갑시키지 않는다.
- 기본 선택 규칙은 `anchor_soft_guard`; 이것은 검증 전 시작 규칙이다. 자동 선택은 `fit_oof`를 명시해야 작동한다.
- 전체 validation의 정답을 사용해 fitting한 점수를 독립 성적으로 보고하지 않는다. meta-holdout은 선택 전에 분리하고 학습에 쓰지 않는다.

**검증 범위:** 셀 문법, CPU 합성 fixture의 ID/클래스 정렬·누수 방지·확률 결합·OOF·CSV/NPY 저장. **미실행:** 실제 모델별 대회 예측을 입력한 앙상블 및 Public/Private 채점. 기존 RF_v1/A를 재현했다고 주장하지 않는다.

처음 실행: 01 설치(필요 시) → 20 `CFG["models"]`·정답 CSV·sample submission 경로 등록 → `inspect` → `evaluate` → `export`.


## 01. 환경 준비

앙상블은 저장된 예측을 결합하므로 VLM을 다시 로드하지 않는다. NumPy·pandas·SciPy·scikit-learn을 사용한다. CPU에서 실행 가능하며 실제 설치 버전을 run에 기록한다. 설치 셀은 기본 꺼짐이다.


In [ ]:
import sys, subprocess
INSTALL_PACKAGES = False
# 이 작성 환경에서 CPU 검증한 버전. 기존 환경을 자동으로 교체하지 않는다.
PACKAGES = ["numpy==2.3.5", "pandas==2.2.3", "scipy==1.17.0", "scikit-learn==1.8.0"]
if INSTALL_PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *PACKAGES])
    print("설치 후 커널을 재시작하고 다음 셀부터 실행하세요.")


## 02. 라이브러리·공통 함수

JSON/CSV/NPY를 읽고 해시·환경·원자적 저장을 준비한다. NPY는 `allow_pickle=False`로 읽는다. ID를 행 번호로 가정하지 않는다.


In [ ]:
from __future__ import annotations
import argparse
from copy import deepcopy
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
import hashlib, html, importlib.metadata, json, math, os, platform, tempfile
from typing import Any
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CLASSES = ['a', 'b', 'c', 'd']
VERSION = '2026-10-05 v1'

def jsonable(value):
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return jsonable(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

def digest(value):
    raw = json.dumps(jsonable(value), ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def hash_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(jsonable(value), ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    os.replace(temp, path)

def save_npy(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    with temp.open('wb') as f:
        np.save(f, np.asarray(value), allow_pickle=False)
    os.replace(temp, path)

def save_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    frame.to_csv(temp, index=False, encoding='utf-8-sig')
    os.replace(temp, path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))

def environment():
    return {'python': platform.python_version(), 'platform': platform.platform(),
            'versions': {name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'scipy', 'scikit-learn']},
            'gpu_required': False, 'version': VERSION}


## 03. 전체 설정·초기값

예시 모델 경로는 실제 경로로 교체해야 한다. 검사만 하는 기본 모드와 학습·export 모드를 분리한다. 옵션과 상세 초기값은 펼친 JSON에도 저장된다.


In [ ]:
def default_config():
    return {
        'schema_version': 1, 'classes': CLASSES.copy(), 'seed': 42,
        'models': [], 'validation_csv': None, 'sample_submission_csv': None,
        'validation_columns': {'id': 'id', 'answer': 'answer', 'group': None},
        'validation_is_gold_confirmed': False,
        'sample_submission_columns': {'id': 'id', 'answer': 'answer'},
        'output_root': 'ensemble_runs', 'run_id': None,
        'strict': {'reject_smoke': True, 'reject_onehot_like_soft': True,
                   'allow_unique_id_groups': False,
                   'require_heldout_confirmation': True, 'require_task_confirmation_if_no_rows': True,
                   'probability_sum_tolerance': 0.002},
        'meta_split': {'holdout_fraction': 0.2, 'method': 'StratifiedGroupKFold',
                       'manifest_path': None, 'n_inner_folds': 5},
        'anchor': {'name': None, 'selection': 'best_on_meta_fit'},
        'calibration': {'mode': 'none', 'temperature_bounds': [0.5, 3.0], 'ece_bins': 10},
        'soft': {'fixed_weights': {}, 'use_calibrated': True},
        'hard': {'fixed_weights': {}, 'tie_break': 'anchor_if_tied_then_abcd'},
        'learned_weights': {'enabled': False, 'l2_to_uniform': 0.01, 'maxiter': 300},
        'guard': {'min_supporters_excluding_anchor': 2, 'min_proposal_confidence': 0.8,
                  'min_proposal_margin': 0.15, 'max_anchor_confidence': 0.9,
                  'min_vote_fraction': 0.6, 'proposal': 'soft_fixed'},
        'core': {'names': [], 'auto_size': 3, 'exclude_anchor_from_auto': True, 'unanimity_sizes': [4, 8]},
        'meta_models': {
            'logistic': {'enabled': False, 'C': 0.3, 'max_iter': 1000},
            'rf': {'enabled': False, 'n_estimators': 300, 'max_depth': 6,
                   'min_samples_leaf': 12, 'max_features': 'sqrt', 'n_jobs': 2},
            'features': {'include_soft_log_probs': True, 'include_hard_indicators': True}},
        'selection': {'policy': 'fixed', 'fixed_recipe': 'anchor_soft_guard',
                      'fallback_recipe': 'anchor', 'candidates': [],
                      'min_fit_oof_gain_count': 1, 'max_fit_oof_harm_count': None,
                      'force_inner_oof': False, 'refit_on_all_validation': False},
        'test_only_recipes': [],
        'manual_review': {'enabled': False, 'changes_csv': None, 'require_reason': True,
                          'export_queue': True, 'queue_limit': None},
        'artifacts': {'save_all_candidate_submissions': True, 'write_charts': True,
                      'save_aligned_inputs': False},
        'evidence': {'template_authoring_real_competition_predictions_executed': False,
                     'public_private': None, 'reproduces_historical_RF_v1_or_A': False},
    }

def validate_config(cfg):
    if cfg['classes'] != CLASSES:
        raise ValueError('출력 canonical 클래스는 a/b/c/d 순서여야 합니다.')
    names = [m['name'] for m in cfg['models']]
    if len(set(names)) != len(names):
        raise ValueError('모델 이름이 중복입니다.')
    for name in names:
        if not name or any(x in name for x in '/\\\n\r'):
            raise ValueError('모델 이름에는 경로·개행을 넣지 마세요.')
    if cfg['calibration']['mode'] not in {'none', 'final_probs_temperature'}:
        raise ValueError('온도 보정은 none / final_probs_temperature만 지원합니다. raw logits와 혼동하지 마세요.')
    bounds = cfg['calibration']['temperature_bounds']
    if len(bounds) != 2 or not 0 < bounds[0] < bounds[1]:
        raise ValueError('온도 범위가 올바르지 않습니다.')
    if cfg['selection']['policy'] not in {'fixed', 'fit_oof'}:
        raise ValueError('선택은 fixed / fit_oof만 지원합니다.')
    if cfg['selection']['refit_on_all_validation']:
        raise ValueError('v1은 meta-holdout을 fitting에 쓰지 않습니다. refit하려면 별도 독립 평가를 먼저 준비하세요.')
    fraction = cfg['meta_split']['holdout_fraction']
    if not 0 < fraction < 1:
        raise ValueError('holdout fraction은 0과 1 사이여야 합니다.')
    k = round(1 / fraction)
    if k < 2 or abs(fraction - 1/k) > 1e-9:
        raise ValueError('group split은 1/k holdout을 지원합니다. 기본 0.2입니다.')
    if cfg['meta_split']['n_inner_folds'] < 2:
        raise ValueError('inner fold는 최소 2입니다.')
    g = cfg['guard']
    if g['min_supporters_excluding_anchor'] < 1:
        raise ValueError('기준 모델 외 지지 수는 1 이상이어야 합니다.')
    for key in ['min_proposal_confidence', 'min_proposal_margin', 'max_anchor_confidence', 'min_vote_fraction']:
        if not 0 <= g[key] <= 1:
            raise ValueError(f'guard {key}는 0~1이어야 합니다.')


## 04. ID·정답·클래스 검증

ID 중복·누락, a/b/c/d 열 순서, 정답 매핑을 검사한다. 문자열 ID의 앞자리 0을 보존한다. 클래스 순서와 보기 원래 순서 복원이 명시되지 않은 soft 파일은 중단한다.


In [ ]:
def normalize_id(value):
    if isinstance(value, (int, np.integer)) and not isinstance(value, (bool, np.bool_)):
        return str(int(value))
    if not isinstance(value, str) or not value or value.strip() != value:
        raise ValueError(f'ID가 비어 있거나 문자열/정수가 아닙니다: {value!r}')
    return value

def check_ids(ids, context):
    ids = [normalize_id(v) for v in ids]
    if not ids or len(set(ids)) != len(ids):
        raise ValueError(f'{context}: 빈 ID 또는 ID 중복')
    return ids

def read_ids(path):
    values = read_json(path)
    if isinstance(values, dict):
        values = values.get('ids')
    if not isinstance(values, list):
        raise ValueError(f'{path}: ID JSON은 목록 또는 {{"ids": [...]}}여야 합니다.')
    return check_ids(values, str(path))

def read_table(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')

def answers_to_indices(values, context):
    result = []
    for v in values:
        if isinstance(v, (int, np.integer)) and not isinstance(v, (bool, np.bool_)):
            if not 0 <= int(v) < 4:
                raise ValueError(f'{context}: label 인덱스 범위 오류')
            result.append(int(v))
        elif isinstance(v, str) and v.strip().lower() in CLASSES:
            result.append(CLASSES.index(v.strip().lower()))
        else:
            raise ValueError(f'{context}: 정답은 a/b/c/d 또는 0~3 정수여야 합니다. 숫자 CSV는 명시적 매핑이 필요합니다: {v!r}')
    return np.asarray(result, dtype=np.int64)

def class_permutation(order):
    if not isinstance(order, list) or len(order) != 4 or set(order) != set(CLASSES):
        raise ValueError('soft 열의 class_order를 명시하세요. a/b/c/d를 한 번씩 포함해야 합니다.')
    return [order.index(c) for c in CLASSES]

def align_array(values, source_ids, target_ids, context):
    if len(values) != len(source_ids):
        raise ValueError(f'{context}: ID와 행 수가 다릅니다.')
    if set(source_ids) != set(target_ids):
        missing = sorted(set(target_ids)-set(source_ids))[:5]
        extra = sorted(set(source_ids)-set(target_ids))[:5]
        raise ValueError(f'{context}: 평가 집합이 다릅니다. missing={missing}, extra={extra}; 교집합 자동 비교를 하지 않습니다.')
    lookup = {v: i for i, v in enumerate(source_ids)}
    return np.asarray(values)[[lookup[v] for v in target_ids]]

def check_probs(p, n, context, cfg):
    p = np.asarray(p, dtype=np.float64)
    if p.shape != (n, 4) or not np.isfinite(p).all() or np.any(p < 0) or np.any(p > 1.000001):
        raise ValueError(f'{context}: 유한한 (N,4) 확률이 아닙니다.')
    sums = p.sum(1)
    if np.any(abs(sums - 1) > cfg['strict']['probability_sum_tolerance']):
        raise ValueError(f'{context}: 확률 행 합이 1이 아닙니다. logits는 probs로 등록하지 마세요.')
    if cfg['strict']['reject_onehot_like_soft'] and np.all((p == 0) | (p == 1)):
        raise ValueError(f'{context}: 전체가 one-hot입니다. hard 답을 실제 soft로 등록할 수 없습니다.')
    if np.any(sums <= 0):
        raise ValueError(f'{context}: 0 합 확률')
    # 저장 반올림 오차만 보정하고 원본 해시/정규화 여부를 감사한다.
    return p / sums[:, None]


## 05. 모델별 soft/hard 로드

soft run의 최종 `valid_probs.npy`/`test_probs.npy`를 기본으로 쓴다. TTA 평균 대신 shift0 logits를 덮어쓰지 않는다. 하드 CSV, 외부 NPY, test-only 입력은 별도 선언이 필요하다.


In [ ]:
@dataclass
class ModelInput:
    name: str
    kind: str
    role: str
    valid_ids: list[str] | None = None
    test_ids: list[str] | None = None
    valid_probs: np.ndarray | None = None
    test_probs: np.ndarray | None = None
    valid_pred: np.ndarray | None = None
    test_pred: np.ndarray | None = None
    valid_labels: np.ndarray | None = None
    valid_rows: pd.DataFrame | None = None
    train_rows: pd.DataFrame | None = None
    validation_signature: str | None = None
    spec: dict = field(default_factory=dict)
    audit: dict = field(default_factory=dict)

def resolve_path(spec, key, relative=None):
    if spec.get(key):
        return Path(spec[key]).expanduser().resolve()
    if spec.get('run_dir') and relative:
        return (Path(spec['run_dir']).expanduser().resolve()/relative)
    return None

def load_model(spec, cfg, need_test):
    name, kind, role = spec['name'], spec['kind'], spec.get('role', 'complete')
    if role not in {'complete', 'test_only'} or kind not in {'soft_run', 'soft_files', 'hard_csv'}:
        raise ValueError(f'{name}: kind/role 오류')
    if role == 'test_only' and not need_test:
        return ModelInput(name, kind, role, spec=deepcopy(spec), audit={'status':'test_only_not_loaded_without_test'})
    if role == 'complete' and cfg['strict']['require_heldout_confirmation'] and spec.get('heldout_predictions_confirmed') is not True:
        raise ValueError(f'{name}: heldout_predictions_confirmed=True로 원 VLM 학습에서 제외된 validation임을 확인해야 합니다.')
    model = ModelInput(name, kind, role, spec=deepcopy(spec), audit={'files':{}, 'input_kind':kind, 'role':role})
    def use(path, required=True):
        if path is None or not path.is_file():
            if required:
                raise FileNotFoundError(f'{name}: 필수 파일 없음: {path}')
            return False
        model.audit['files'][str(path)] = {'sha256':hash_file(path), 'bytes':path.stat().st_size}
        return True
    splits = ([] if role == 'test_only' else ['valid']) + (['test'] if need_test else [])
    if kind.startswith('soft'):
        meta_path = resolve_path(spec, 'metadata', 'probs/probability_metadata.json')
        meta = read_json(meta_path) if use(meta_path, False) else {}
        model.audit['probability_metadata']=deepcopy(meta)
        for split in splits:
            p_path = resolve_path(spec, f'{split}_probs', f'probs/{split}_probs.npy')
            ids_path = resolve_path(spec, f'{split}_ids', f'probs/{split}_ids.json')
            use(p_path); use(ids_path)
            ids = read_ids(ids_path)
            info = meta.get(split, {})
            if split=='valid' and info.get('meta_in_sample_present'):
                raise ValueError(f'{name}: full validation에 메타 학습 in-sample 예측이 포함됩니다. 상위 앙상블에는 별도의 OOF/holdout 입력을 준비하세요.')
            if cfg['strict']['reject_smoke'] and (info.get('is_smoke') or 'SMOKE' in p_path.name):
                raise ValueError(f'{name}: smoke 결과는 제출용 입력이 아닙니다.')
            if info.get('hard_onehot_used') is True or spec.get('hard_as_onehot') is True:
                raise ValueError(f'{name}: hard one-hot은 soft 입력이 아닙니다.')
            order = info.get('class_order', spec.get('class_order'))
            perm = class_permutation(order)
            # 앞 노트북의 restored original-order 기록도 인정한다.
            restored = (info.get('original_option_order') is True or spec.get('original_option_order') is True
                        or (('restored' in info.get('probs_semantics','') or 'restored' in info.get('logits_semantics',''))
                            and bool(info.get('shifts'))))
            if not restored:
                raise ValueError(f'{name}/{split}: original_option_order=True 또는 원래 보기 순서 복원 metadata가 필요합니다.')
            if not info and spec.get('confirmed_real_soft') is not True:
                raise ValueError(f'{name}: metadata 없는 외부 NPY는 confirmed_real_soft=True가 필요합니다.')
            if info.get('hard_onehot_used') is None and spec.get('confirmed_real_soft') is not True:
                raise ValueError(f'{name}: soft 출처가 미확인입니다.')
            p = check_probs(np.load(p_path, allow_pickle=False), len(ids), f'{name}/{split}', cfg)[:,perm]
            setattr(model, f'{split}_ids', ids)
            setattr(model, f'{split}_probs', p)
            setattr(model, f'{split}_pred', p.argmax(1))
            model.audit[split] = {'n':len(ids), 'original_class_order':order, 'column_permutation':perm,
                'probs_semantics':info.get('probs_semantics',spec.get('probs_semantics','user_confirmed_real_soft')),
                'softmax_reference_logits_used':False, 'raw_logits_consumed':False,
                'normalized_rounding_error':bool(np.any(abs(np.load(p_path,allow_pickle=False).sum(1)-1)>1e-7))}
        if role == 'complete':
            labels_path = resolve_path(spec,'valid_labels','probs/valid_labels.npy')
            if use(labels_path, False):
                raw_labels = np.load(labels_path,allow_pickle=False)
                if raw_labels.shape != (len(model.valid_ids),) or not np.issubdtype(raw_labels.dtype,np.integer):
                    raise ValueError(f'{name}: valid_labels는 (N,) 정수 인덱스여야 합니다.')
                if np.any((raw_labels<0)|(raw_labels>=4)):
                    raise ValueError(f'{name}: label 범위 오류')
                order=meta.get('valid',{}).get('class_order',spec.get('class_order'))
                model.valid_labels=answers_to_indices([order[int(v)] for v in raw_labels],name)
    else:
        for split in splits:
            path=resolve_path(spec,f'{split}_csv')
            use(path)
            frame=read_table(path)
            id_col=spec.get('id_column','id')
            pred_col=spec.get(f'{split}_answer_column','pred' if split=='valid' else 'answer')
            if id_col not in frame or pred_col not in frame:
                raise ValueError(f'{name}/{split}: ID/예측 열 없음')
            ids=check_ids(frame[id_col].tolist(),name)
            mapping=spec.get('answer_value_map',{})
            preds=answers_to_indices([mapping.get(x,x) for x in frame[pred_col]],name)
            setattr(model,f'{split}_ids',ids);setattr(model,f'{split}_pred',preds)
            model.audit[split]={'n':len(ids),'kind':'hard_answer_only','real_soft_available':False}
            label_col=spec.get('valid_gold_column') if split=='valid' else None
            if label_col:
                model.valid_labels=answers_to_indices([mapping.get(x,x) for x in frame[label_col]],name)
    if role=='complete':
        rows_path=resolve_path(spec,'validation_rows','validation_rows.csv')
        if use(rows_path,False):
            model.valid_rows=read_table(rows_path)
        train_path=resolve_path(spec,'training_rows','train_base_rows.csv')
        if use(train_path,False):
            model.train_rows=read_table(train_path)
        summary_path=resolve_path(spec,'validation_summary','validation_summary.json')
        if use(summary_path,False):
            summary=read_json(summary_path)
            if cfg['strict']['reject_smoke'] and summary.get('is_smoke'):
                raise ValueError(f'{name}: validation summary가 smoke입니다.')
            model.validation_signature=summary.get('validation_signature')
    for key,relative in [('source_config','config.json'),('source_model_metadata','model_metadata.json')]:
        use(resolve_path(spec,key,relative),False)
    return model


## 06. 공통 validation·test 정렬

기준 정답 CSV 또는 첫 run의 `validation_rows.csv`에서 ID/정답/이미지 그룹을 만든다. 가능한 문항·보기·이미지 해시와 학습 누수를 대조한다. 같은 ID라도 정답이나 내용이 다르면 중단한다.


In [ ]:
def canonical_validation(models,cfg):
    complete=[m for m in models if m.role=='complete']
    if not complete:
        raise ValueError('공통 validation을 가진 complete 모델이 최소 하나 필요합니다.')
    column=cfg['validation_columns']
    if cfg['validation_csv']:
        rows=read_table(cfg['validation_csv'])
    else:
        rows=next((m.valid_rows.copy() for m in complete if m.valid_rows is not None),None)
    if rows is None:
        first=next((m for m in complete if m.valid_labels is not None),None)
        if first is None:
            raise ValueError('validation 정답 CSV 또는 valid_labels.npy가 필요합니다.')
        rows=pd.DataFrame({'id':first.valid_ids,'answer':[CLASSES[i] for i in first.valid_labels]})
        column={'id':'id','answer':'answer','group':column['group']}
    if column['id'] not in rows or column['answer'] not in rows:
        raise ValueError('validation CSV의 id/answer 열을 확인하세요.')
    rows=rows.rename(columns={column['id']:'id',column['answer']:'answer'})
    ids=check_ids(rows.id.tolist(),'canonical validation')
    labels=answers_to_indices(rows.answer.tolist(),'canonical validation')
    rows['answer']=[CLASSES[i] for i in labels]
    if '_src' in rows:
        if set(rows._src)!={'train_gold'}:
            raise ValueError('평가 정답에 dev/pseudo label이 포함되었습니다. train_gold만 사용하세요.')
    elif not cfg['validation_is_gold_confirmed']:
        raise ValueError('출처 열이 없으면 validation_is_gold_confirmed=True로 정답 출처를 확인해야 합니다.')
    group_col=column['group']
    if group_col is None:
        group_col=next((c for c in ['_group','image_group','_hash','image_sha256','path'] if c in rows),None)
    if group_col:
        if group_col not in rows or (rows[group_col]=='').any():
            raise ValueError('이미지 그룹 열이 없거나 빈 값입니다.')
        groups=rows[group_col].to_numpy(dtype=str)
    else:
        groups=np.asarray(ids)
    tasks=[c for c in ['question',*CLASSES,'_hash'] if c in rows]
    expected_sig=next((m.validation_signature for m in complete if m.validation_signature),None)
    warnings=[]
    for m in complete:
        m.valid_pred=align_array(m.valid_pred,m.valid_ids,ids,m.name+'/valid')
        if m.valid_probs is not None:
            m.valid_probs=align_array(m.valid_probs,m.valid_ids,ids,m.name+'/valid soft')
        if m.valid_labels is not None:
            lab=align_array(m.valid_labels,m.valid_ids,ids,m.name+'/labels')
            if not np.array_equal(lab,labels):
                raise ValueError(f'{m.name}: 같은 ID의 정답이 다릅니다.')
            m.valid_labels=lab
        m.valid_ids=ids.copy()
        signature_differs=bool(expected_sig and m.validation_signature and expected_sig!=m.validation_signature)
        if m.valid_rows is not None:
            other=m.valid_rows
            if 'id' not in other or 'answer' not in other:
                raise ValueError(f'{m.name}: validation_rows의 id/answer 열이 없습니다.')
            other_ids=check_ids(other.id.tolist(),m.name+'/validation_rows')
            other=other.set_index('id').loc[ids].reset_index() if set(other_ids)==set(ids) else None
            if other is None:
                raise ValueError(f'{m.name}: validation_rows와 예측 집합이 다릅니다.')
            if not np.array_equal(answers_to_indices(other.answer,m.name),labels):
                raise ValueError(f'{m.name}: validation_rows 정답 불일치')
            for c in tasks:
                if c not in other or not np.array_equal(rows[c].to_numpy(),other[c].to_numpy()):
                    raise ValueError(f'{m.name}: validation 문항/보기/이미지 내용 불일치: {c}')
            if group_col and group_col in other and not np.array_equal(groups,other[group_col].to_numpy(dtype=str)):
                raise ValueError(f'{m.name}: 이미지 그룹 정의가 다릅니다.')
        elif cfg['strict']['require_task_confirmation_if_no_rows'] and m.spec.get('same_task_confirmed') is not True:
            raise ValueError(f'{m.name}: validation_rows가 없습니다. 공통 문항·보기·이미지를 대조한 뒤 same_task_confirmed=True를 지정하세요.')
        else:
            warnings.append(f'{m.name}: 실제 문항 CSV 부재, 동일 문항 여부는 사용자 확인에 의존')
        if signature_differs:
            if m.valid_rows is not None and set(['question',*CLASSES,'_hash'])<=set(tasks):
                warnings.append(f'{m.name}: 원 validation signature는 다르지만 ID 정렬 후 정답·질문·보기·이미지 해시 일치를 확인했습니다. 원 서명 차이를 inputs에 기록했습니다.')
            else:
                raise ValueError(f'{m.name}: validation_signature가 다르고 충분한 문항/이미지 CSV 대조가 없습니다.')
        if m.train_rows is not None:
            training=m.train_rows
            if 'id' not in training:
                raise ValueError(f'{m.name}: train_base_rows에 ID 없음')
            if set(training.id)&set(ids):
                raise ValueError(f'{m.name}: 원 모델 학습과 validation ID 누수')
            for c in ['_group','_hash']:
                if c in rows and c in training and set(rows[c])&set(training[c]):
                    raise ValueError(f'{m.name}: 원 모델 학습과 validation 이미지 누수: {c}')
        else:
            warnings.append(f'{m.name}: 학습 목록 부재, 원 VLM 학습 제외 여부는 declared heldout 확인에 의존')
    if not tasks:
        warnings.append('문항/보기/이미지 해시 없이 ID·정답만 대조했습니다.')
    signature=digest(rows[['id','answer',*tasks]].to_dict('records'))
    return {'rows':rows,'ids':ids,'labels':labels,'groups':groups,'group_column':group_col,
            'signature':signature,'base_validation_signature':expected_sig,'warnings':warnings}

def canonical_test(models,cfg):
    if not cfg['sample_submission_csv']:
        raise ValueError('export에는 실제 sample_submission_csv 경로가 필요합니다.')
    sample=read_table(cfg['sample_submission_csv'])
    cols=cfg['sample_submission_columns']
    if cols['id'] not in sample or cols['answer'] not in sample:
        raise ValueError('sample_submission 열을 확인하세요.')
    ids=check_ids(sample[cols['id']].tolist(),'sample submission')
    for m in models:
        if m.test_pred is None:
            raise ValueError(f'{m.name}: test 답이 없습니다.')
        m.test_pred=align_array(m.test_pred,m.test_ids,ids,m.name+'/test')
        if m.test_probs is not None:
            m.test_probs=align_array(m.test_probs,m.test_ids,ids,m.name+'/test soft')
        m.test_ids=ids.copy()
    return {'ids':ids,'sample_sha256':hash_file(cfg['sample_submission_csv']),
            'n':len(ids),'prediction_order':'sample_submission_csv original order'}

def input_catalog(models):
    return [{'name':m.name,'kind':m.kind,'role':m.role,
             'n_valid':None if m.valid_ids is None else len(m.valid_ids),
             'n_test':None if m.test_ids is None else len(m.test_ids),
             'actual_soft_valid':m.valid_probs is not None,'actual_soft_test':m.test_probs is not None,
             'used_for_meta_learning':m.role=='complete'} for m in models]


## 07. 이미지 그룹 meta 분할

이미지 그룹 단위로 기본 4:1 meta-fit/meta-holdout을 만든다. 이것은 원래 모델의 train/validation 4:1과 별개다. 원본 validation의 클래스 분포를 유지하려 하며 평가 정답을 강제로 25%씩 만들지 않는다.


In [ ]:
def grouped_folds(labels,groups,k,seed):
    if len(np.unique(groups))<k:
        raise ValueError(f'{k} folds를 만들 이미지 그룹이 부족합니다.')
    if min(np.bincount(labels,minlength=4))<k:
        raise ValueError(f'각 정답 클래스가 {k}개 이상이어야 합니다. fold 수를 줄이세요.')
    splitter=StratifiedGroupKFold(n_splits=k,shuffle=True,random_state=seed)
    result=[]
    for tr,va in splitter.split(np.zeros(len(labels)),labels,groups):
        if set(groups[tr])&set(groups[va]):
            raise AssertionError('meta group 누수')
        if set(labels[tr])!={0,1,2,3} or set(labels[va])!={0,1,2,3}:
            raise ValueError('어떤 fold에 정답 클래스가 없습니다. 그룹 분포·fold 수를 확인하세요.')
        result.append((tr,va))
    return result

def make_meta_split(canonical,cfg):
    if canonical['group_column'] is None and not cfg['strict']['allow_unique_id_groups']:
        raise ValueError('평가에는 이미지 그룹이 필요합니다. 이전 run의 validation_rows.csv를 연결하세요. ID별 그룹 예외는 같은 이미지가 없음을 확인한 경우에만 허용하세요.')
    y,g,ids=canonical['labels'],canonical['groups'],canonical['ids']
    sig=digest({'validation':canonical['signature'],'groups':g.tolist(),'seed':cfg['seed'],
                'fraction':cfg['meta_split']['holdout_fraction']})
    existing=cfg['meta_split']['manifest_path']
    if existing:
        info=read_json(existing)
        if info['signature']!=sig:
            raise ValueError('meta split manifest 서명이 다릅니다.')
        fit_ids,hold_ids=info['fit_ids'],info['holdout_ids']
        if set(fit_ids)&set(hold_ids) or set(fit_ids)|set(hold_ids)!=set(ids):
            raise ValueError('meta split ID 누락/중복')
        fit_ids=check_ids(fit_ids,'meta fit manifest');hold_ids=check_ids(hold_ids,'meta holdout manifest')
        lookup={v:i for i,v in enumerate(ids)}
        fit=np.array([lookup[v] for v in fit_ids],dtype=int);hold=np.array([lookup[v] for v in hold_ids],dtype=int)
    else:
        k=round(1/cfg['meta_split']['holdout_fraction'])
        dist=np.bincount(y,minlength=4)/len(y)
        choices=[]
        for fit,hold in grouped_folds(y,g,k,cfg['seed']):
            d=np.bincount(y[hold],minlength=4)/len(hold)
            loss=abs(len(hold)/len(y)-1/k)+abs(d-dist).mean()
            choices.append((float(loss),fit,hold))
        _,fit,hold=min(choices,key=lambda x:x[0])
        info={'signature':sig,'fit_ids':[ids[i] for i in fit],'holdout_ids':[ids[i] for i in hold],
              'method':'StratifiedGroupKFold_choose_distribution_fold','seed':cfg['seed'],
              'requested_holdout_fraction':cfg['meta_split']['holdout_fraction'],
              'realized_holdout_fraction':len(hold)/len(y),'group_column':canonical['group_column']}
    if set(g[fit])&set(g[hold]):
        raise ValueError('manifest의 fit/holdout 이미지 그룹 누수')
    return fit,hold,info


## 08. 평가 지표·Rescue/Harm

accuracy·NLL·Brier·ECE·클래스별 정확도와 기준 모델 대비 rescue/harm을 계산한다. 투표율에는 NLL/ECE를 실제 확률처럼 적용하지 않는다.


In [ ]:
def confidence_margin(p):
    sorted_p=np.sort(p,axis=1)
    return sorted_p[:,-1],sorted_p[:,-1]-sorted_p[:,-2]

def metrics(pred,y,probs=None,bins=10):
    correct=np.asarray(pred)==np.asarray(y)
    out={'n':len(y),'accuracy':float(correct.mean()),'correct':int(correct.sum()),
         'by_class':{c:{'n':int((y==i).sum()),'accuracy':float(correct[y==i].mean()) if (y==i).any() else None}
                     for i,c in enumerate(CLASSES)}}
    if probs is not None:
        clipped=np.clip(probs,1e-15,1)
        out['nll']=float(-np.log(clipped[np.arange(len(y)),y]).mean())
        target=np.eye(4)[y]
        out['brier']=float(np.square(probs-target).sum(1).mean())
        conf=probs.max(1)
        ece=0.0
        for k in range(bins):
            mask=(conf>=k/bins)&(conf<(k+1)/bins if k<bins-1 else conf<=1)
            if mask.any():
                ece+=mask.mean()*abs(correct[mask].mean()-conf[mask].mean())
        out['ece']=float(ece)
    return out

def rescue_harm(pred,anchor,y):
    a=anchor==y;b=pred==y
    rescue=int((~a&b).sum());harm=int((a&~b).sum())
    return {'changed':int((pred!=anchor).sum()),'rescue':rescue,'harm':harm,'net':rescue-harm}

def pairwise_comparison(models,y):
    rows=[]
    for i,a in enumerate(models):
        for b in models[i+1:]:
            ca=a.valid_pred==y;cb=b.valid_pred==y
            rows.append({'anchor':a.name,'other':b.name,'n':len(y),
                         'disagreement':int((a.valid_pred!=b.valid_pred).sum()),
                         'both_wrong':int((~ca&~cb).sum()),
                         'error_jaccard':float((~ca&~cb).sum()/(~ca|~cb).sum()) if (~ca|~cb).any() else 0.,
                         **rescue_harm(b.valid_pred,a.valid_pred,y)})
    return pd.DataFrame(rows,columns=['anchor','other','n','disagreement','both_wrong','error_jaccard','changed','rescue','harm','net'])


## 09. 온도·soft 가중치 학습

기본 보정 없음. 선택 시 최종 TTA 확률의 log 값을 재정규화해 온도를 학습한다. 이는 원래 모델 logits 보정과 다르다. 가중치는 meta-fit만 사용하고 uniform 방향 정규화를 둔다.


In [ ]:
def temperature_probs(p,temperature):
    # log(p)로 만든 보정 입력이지 실제 VLM raw logits가 아니다.
    z=np.log(np.clip(p,1e-15,1))/float(temperature)
    z-=z.max(1,keepdims=True)
    q=np.exp(z)
    return q/q.sum(1,keepdims=True)

def fit_temperature(p,y,cfg):
    if cfg['calibration']['mode']=='none':
        return 1.0
    def loss(t):
        q=temperature_probs(p,t)
        return float(-np.log(np.clip(q[np.arange(len(y)),y],1e-15,1)).mean())
    result=minimize_scalar(loss,bounds=cfg['calibration']['temperature_bounds'],method='bounded')
    if not result.success or not np.isfinite(result.x):
        raise RuntimeError('온도 최적화 실패')
    # T=1보다 NLL이 나빠지는 수치 최적화 결과는 쓰지 않는다.
    return float(result.x) if loss(result.x)<loss(1.0) else 1.0

def fixed_weights(names,specified):
    unknown=set(specified)-set(names)
    if unknown:
        raise ValueError(f'가중치 대상 이름을 확인하세요: {sorted(unknown)}')
    w=np.asarray([specified.get(n,1.) for n in names],dtype=float)
    if not np.isfinite(w).all() or (w<0).any() or w.sum()<=0:
        raise ValueError('가중치는 유한한 0 이상이어야 하고 합이 양수여야 합니다.')
    return w/w.sum()

def fit_soft_weights(stack,y,cfg):
    m=stack.shape[0]
    uniform=np.full(m,1/m)
    if m==1 or not cfg['learned_weights']['enabled']:
        return uniform
    target=stack[:,np.arange(len(y)),y]
    penalty=cfg['learned_weights']['l2_to_uniform']
    def loss(w):
        gold=np.clip(w@target,1e-15,1)
        return float(-np.log(gold).mean()+penalty*np.square(w-uniform).sum())
    result=minimize(loss,uniform,method='SLSQP',bounds=[(0,1)]*m,
                    constraints=[{'type':'eq','fun':lambda w:w.sum()-1}],
                    options={'maxiter':cfg['learned_weights']['maxiter'],'ftol':1e-10})
    if not result.success or not np.isfinite(result.x).all():
        raise RuntimeError('soft 가중치 최적화 실패: '+str(result.message))
    w=np.maximum(result.x,0)
    return w/w.sum()

def choose_anchor(models,idx,y,cfg):
    name=cfg['anchor']['name']
    if name:
        if name not in [m.name for m in models]:
            raise ValueError('anchor는 complete 모델이어야 합니다.')
        return name
    if cfg['anchor']['selection']!='best_on_meta_fit':
        raise ValueError('anchor 자동 선택은 best_on_meta_fit만 지원합니다.')
    # 동점은 등록 순서. holdout 정답이나 Public 점수는 쓰지 않는다.
    return max(models,key=lambda m:float((m.valid_pred[idx]==y[idx]).mean())).name

def rank_models(models,idx,y):
    return [m.name for m in sorted(models,key=lambda m:-float((m.valid_pred[idx]==y[idx]).mean()))]


## 10. 투표·만장일치·기준 모델 보호

hard ties는 기준 모델이 최대 득표 클래스에 포함되면 기준 모델을 따른다. 그 밖에는 canonical a/b/c/d 순서로 결정한다. 조건부 교체는 확률·margin·기준 모델 외 지지 수를 모두 확인한다. 서로 다른 run이 통계적으로 독립임을 보장하지 않는다.


In [ ]:
def vote_counts(predictions,weights=None):
    votes=np.asarray(predictions,dtype=int)
    w=np.ones(len(votes)) if weights is None else np.asarray(weights)
    return np.stack([(w[:,None]*(votes==c)).sum(0) for c in range(4)],axis=1)

def resolve_tie(counts,anchor):
    maxima=counts.max(1,keepdims=True)
    tied=np.isclose(counts,maxima,atol=1e-12,rtol=0)
    pred=counts.argmax(1)
    choose=tied[np.arange(len(anchor)),anchor]
    pred[choose]=anchor[choose]
    return pred

def guard_mask(proposal,anchor,soft,anchor_soft,supporters,votes,cfg,with_vote):
    conf,margin=confidence_margin(soft)
    anchor_conf=anchor_soft.max(1)
    support=(supporters==proposal[None,:]).sum(0) if len(supporters) else np.zeros(len(proposal),dtype=int)
    g=cfg['guard']
    mask=(proposal!=anchor)&(support>=g['min_supporters_excluding_anchor'])
    mask&=(conf>=g['min_proposal_confidence'])&(margin>=g['min_proposal_margin'])&(anchor_conf<=g['max_anchor_confidence'])
    if with_vote:
        counts=vote_counts(votes)
        voted=resolve_tie(counts,anchor)
        mask&=(voted==proposal)&(counts.max(1)/len(votes)>=g['min_vote_fraction'])
    return mask


## 11. Logistic/RF 선택형 메타 모델

기본 꺼짐. 실제 확률 feature와 hard 범주 feature를 분리해 기록한다. 메타 모델 확률은 새로운 분류기의 출력이며 VLM token 확률이라고 설명하지 않는다. 기존 RF_v1의 복제본이 아니다.


In [ ]:
def meta_features(models,split,idx,temperatures,cfg):
    pieces=[];columns=[]
    for m in models:
        p=getattr(m,split+'_probs')
        pred=getattr(m,split+'_pred')[idx]
        if p is not None:
            q=temperature_probs(p[idx],temperatures.get(m.name,1.)) if cfg['soft']['use_calibrated'] else p[idx]
            pieces.append(q);columns.extend([f'{m.name}:real_soft_{c}' for c in CLASSES])
            if cfg['meta_models']['features']['include_soft_log_probs']:
                pieces.append(np.log(np.clip(q,1e-12,1)));columns.extend([f'{m.name}:derived_log_probability_{c}' for c in CLASSES])
            conf,margin=confidence_margin(q)
            entropy=-(q*np.log(np.clip(q,1e-12,1))).sum(1)
            pieces.append(np.stack([conf,margin,entropy],axis=1))
            columns.extend([f'{m.name}:confidence',f'{m.name}:margin',f'{m.name}:entropy'])
        elif cfg['meta_models']['features']['include_hard_indicators']:
            # 범주형 메타 feature. 이 배열을 model probabilities로 저장하지 않는다.
            pieces.append(np.eye(4)[pred]);columns.extend([f'{m.name}:hard_indicator_{c}' for c in CLASSES])
    votes=np.stack([getattr(m,split+'_pred')[idx] for m in models])
    pieces.append(vote_counts(votes)/len(models));columns.extend([f'vote_fraction_feature_{c}' for c in CLASSES])
    return np.concatenate(pieces,axis=1),columns

def build_meta_model(kind,cfg,seed):
    if kind=='logistic':
        options=cfg['meta_models']['logistic']
        return make_pipeline(StandardScaler(),LogisticRegression(C=options['C'],max_iter=options['max_iter'],random_state=seed))
    options=cfg['meta_models']['rf']
    return RandomForestClassifier(n_estimators=options['n_estimators'],max_depth=options['max_depth'],
          min_samples_leaf=options['min_samples_leaf'],max_features=options['max_features'],n_jobs=options['n_jobs'],random_state=seed)

def model_probabilities(estimator,features):
    q=estimator.predict_proba(features)
    order=list(estimator.classes_)
    if set(order)!={0,1,2,3}:
        raise ValueError('메타 모델 학습 fold에 네 클래스가 모두 필요합니다.')
    return q[:,[order.index(i) for i in range(4)]]

def fit_state(models,idx,y,cfg,anchor,ranking):
    soft=[m for m in models if m.valid_probs is not None]
    temps={m.name:fit_temperature(m.valid_probs[idx],y[idx],cfg) for m in soft}
    qs=[temperature_probs(m.valid_probs[idx],temps[m.name]) if cfg['soft']['use_calibrated'] else m.valid_probs[idx] for m in soft]
    w=fit_soft_weights(np.stack(qs),y[idx],cfg) if soft else None
    state={'anchor':anchor,'ranking':ranking,'temperatures':temps,'soft_names':[m.name for m in soft],
           'learned_weights':w,'estimators':{},'fit_indices':np.asarray(idx,dtype=int),'features':[]}
    if any(cfg['meta_models'][kind]['enabled'] for kind in ['logistic','rf']):
        X,features=meta_features(models,'valid',idx,temps,cfg)
        state['features']=features
        for kind in ['logistic','rf']:
            if cfg['meta_models'][kind]['enabled']:
                estimator=build_meta_model(kind,cfg,cfg['seed'])
                estimator.fit(X,y[idx]);state['estimators'][kind]=estimator
    return state

def state_summary(state,ids):
    return {'anchor':state['anchor'],'ranking':state['ranking'],'temperatures':state['temperatures'],
            'soft_names':state['soft_names'],'learned_weights':state['learned_weights'],
            'meta_models':list(state['estimators']),'feature_columns':state['features'],
            'fit_ids':[ids[i] for i in state['fit_indices']],
            'temperature_semantics':'renormalized final probability power, not raw VLM logits',
            'hard_onehot_semantics':'categorical meta feature only; never an actual input probability'}


## 12. 후보 실행·예측 유형

soft 확률, 메타 모델 확률, hard 답·투표 수를 결과 타입으로 구분한다. 기준 모델 보호 후보의 probabilities는 실제 anchor/proposal 확률 행을 조건별로 고른 것이다.


In [ ]:
@dataclass
class Candidate:
    name: str
    prediction: np.ndarray
    probability: np.ndarray | None = None
    counts: np.ndarray | None = None
    semantics: str = 'hard_answer_only'
    changed: np.ndarray | None = None

def recipe_candidates(models,split,idx,state,cfg):
    by_name={m.name:m for m in models}
    anchor=by_name[state['anchor']]
    ap=getattr(anchor,split+'_pred')[idx]
    raw_anchor=getattr(anchor,split+'_probs')
    aq=None if raw_anchor is None else raw_anchor[idx]
    candidates={'anchor':Candidate('anchor',ap.copy(),None if aq is None else aq.copy(),semantics='actual anchor final probabilities' if aq is not None else 'anchor hard answer')}
    soft=[by_name[n] for n in state['soft_names']]
    soft_probs=[]
    for m in soft:
        p=getattr(m,split+'_probs')[idx]
        soft_probs.append(temperature_probs(p,state['temperatures'][m.name]) if cfg['soft']['use_calibrated'] else p)
    if soft:
        stack=np.stack(soft_probs)
        for name,w in [('soft_mean',np.full(len(soft),1/len(soft))),
                       ('soft_fixed',fixed_weights(state['soft_names'],cfg['soft']['fixed_weights']))]:
            q=np.tensordot(w,stack,axes=(0,0))
            candidates[name]=Candidate(name,q.argmax(1),q,semantics='weighted mean of actual final probabilities; post-calibration if configured')
        if cfg['learned_weights']['enabled']:
            q=np.tensordot(state['learned_weights'],stack,axes=(0,0))
            candidates['soft_learned']=Candidate('soft_learned',q.argmax(1),q,semantics='meta-fit NLL learned probability mean')
    votes=np.stack([getattr(m,split+'_pred')[idx] for m in models])
    counts=vote_counts(votes)
    candidates['hard_vote']=Candidate('hard_vote',resolve_tie(counts,ap),counts=counts,semantics='integer hard vote counts; not soft probabilities')
    if cfg['hard']['fixed_weights']:
        w=fixed_weights([m.name for m in models],cfg['hard']['fixed_weights'])
        counts_w=vote_counts(votes,w)
        candidates['hard_weighted']=Candidate('hard_weighted',resolve_tie(counts_w,ap),counts=counts_w,semantics='weighted hard vote score; not soft probabilities')
    auto_core=[n for n in state['ranking'] if n!=anchor.name or not cfg['core']['exclude_anchor_from_auto']]
    core=cfg['core']['names'] or auto_core[:cfg['core']['auto_size']]
    if set(core)-set(by_name):
        raise ValueError('core 이름은 complete 모델이어야 합니다.')
    groups=[('core_unanimous',core)]
    groups.extend((f'unanimous_{k}',state['ranking'][:k]) for k in cfg['core']['unanimity_sizes'] if len(models)>=k)
    for name,names in groups:
        if len(names)<2:
            continue
        pv=np.stack([getattr(by_name[n],split+'_pred')[idx] for n in names])
        unanimous=(pv==pv[0]).all(0)
        pred=ap.copy();pred[unanimous]=pv[0,unanimous]
        candidates[name]=Candidate(name,pred,counts=vote_counts(pv),semantics='anchor fallback + declared core unanimous hard answer',changed=pred!=ap)
    proposal_name=cfg['guard']['proposal']
    if (proposal_name in candidates and aq is not None and candidates[proposal_name].probability is not None
            and len(models)-1>=cfg['guard']['min_supporters_excluding_anchor']):
        proposal=candidates[proposal_name]
        supporters=np.stack([getattr(m,split+'_pred')[idx] for m in models if m.name!=anchor.name]) if len(models)>1 else np.empty((0,len(idx)),dtype=int)
        for name,with_vote in [('anchor_soft_guard',False),('soft_hard_guard',True)]:
            mask=guard_mask(proposal.prediction,ap,proposal.probability,aq,supporters,votes,cfg,with_vote)
            q=aq.copy();q[mask]=proposal.probability[mask]
            candidates[name]=Candidate(name,q.argmax(1),q,semantics='row-wise routing of actual anchor/proposal probability vectors',changed=mask)
    for kind,estimator in state['estimators'].items():
        X,columns=meta_features(models,split,idx,state['temperatures'],cfg)
        if columns!=state['features']:
            raise AssertionError('메타 feature 순서 변경')
        q=model_probabilities(estimator,X)
        candidates[kind+'_meta']=Candidate(kind+'_meta',q.argmax(1),q,semantics='trained meta classifier predicted probabilities; not VLM logits/probs')
        if aq is not None and len(models)-1>=cfg['guard']['min_supporters_excluding_anchor']:
            supporters=np.stack([getattr(m,split+'_pred')[idx] for m in models if m.name!=anchor.name]) if len(models)>1 else np.empty((0,len(idx)),dtype=int)
            mask=guard_mask(q.argmax(1),ap,q,aq,supporters,votes,cfg,True)
            guarded=aq.copy();guarded[mask]=q[mask]
            candidates[kind+'_guard']=Candidate(kind+'_guard',guarded.argmax(1),guarded,semantics='anchor/meta-model probability routing with vote guard',changed=mask)
    return candidates


## 13. Meta-fit 내부 OOF

옵션 활성화 또는 자동 선택 시 meta-fit 안에서만 그룹별 교차검증한다. 각 fold의 온도·가중치·메타 모델은 해당 fold 학습 부분만 사용한다. 이는 고정 VLM 예측 위의 메타 단계 OOF다.


In [ ]:
def build_fit_oof(models,fit,canonical,cfg):
    y,g=canonical['labels'],canonical['groups']
    folds=grouped_folds(y[fit],g[fit],cfg['meta_split']['n_inner_folds'],cfg['seed']+1)
    predictions={};probabilities={};counts={};seen={};semantics={};audit=[]
    for k,(train_pos,valid_pos) in enumerate(folds):
        train=fit[train_pos];valid=fit[valid_pos]
        fold_anchor=choose_anchor(models,train,y,cfg)
        fold_ranking=rank_models(models,train,y)
        state=fit_state(models,train,y,cfg,fold_anchor,fold_ranking)
        result=recipe_candidates(models,'valid',valid,state,cfg)
        for name,item in result.items():
            if name not in predictions:
                predictions[name]=np.full(len(fit),-1,dtype=int)
                seen[name]=np.zeros(len(fit),dtype=int)
                if item.probability is not None:
                    probabilities[name]=np.full((len(fit),4),np.nan)
                if item.counts is not None:
                    counts[name]=np.full((len(fit),4),np.nan)
                semantics[name]=item.semantics
            predictions[name][valid_pos]=item.prediction;seen[name][valid_pos]+=1
            if item.probability is not None and name in probabilities:
                probabilities[name][valid_pos]=item.probability
            if item.counts is not None and name in counts:
                counts[name][valid_pos]=item.counts
        audit.append({'fold':k,'train_ids':[canonical['ids'][i] for i in train],
                      'oof_ids':[canonical['ids'][i] for i in valid],
                      'state':state_summary(state,canonical['ids'])})
    result={};dropped=[]
    for name,pred in predictions.items():
        if not (seen[name]==1).all():
            dropped.append({'name':name,'reason':'fold별 anchor 유형 변화로 모든 fold에서 실행되지 않은 후보'})
            continue
        p=probabilities.get(name)
        if p is not None and not np.isfinite(p).all():
            dropped.append({'name':name,'reason':'OOF probability 유형 불일치'});continue
        result[name]=Candidate(name,pred,p,counts.get(name),semantics[name])
    return result,{'folds':audit,'dropped_candidates':dropped,'scope':'meta-fit only; no holdout used',
                   'base_vlm_weights_refit_per_fold':False}


## 14. 후보 선택·holdout 동결

기본 고정 레시피는 설정한 guard를 사용하고, 자원이 부족하면 anchor로 후퇴한다. fit_oof 자동 선택은 fit OOF 점수만 사용한다. holdout 결과를 보고 규칙을 반복 변경하면 독립 평가 의미가 약해진다.


In [ ]:
def select_recipe(candidates,oof,fit_labels,cfg):
    fallback=cfg['selection']['fallback_recipe']
    if cfg['selection']['policy']=='fixed':
        desired=cfg['selection']['fixed_recipe']
        name=desired if desired in candidates else fallback
        if name not in candidates:
            raise ValueError('fixed/fallback 레시피를 실행할 수 없습니다.')
        return name,{'policy':'fixed','requested':desired,'selected':name,'holdout_used_for_selection':False,
                     'fallback_used':name!=desired,'reason':'설정에서 고정한 규칙; 미지원이면 명시적 anchor fallback'}
    if not oof or 'anchor' not in oof:
        raise ValueError('fit_oof 선택에는 meta-fit 내부 OOF 결과가 필요합니다.')
    allowed=cfg['selection']['candidates'] or list(oof)
    if set(allowed)-set(oof):
        raise ValueError('선택 후보 일부의 OOF가 없습니다: '+str(sorted(set(allowed)-set(oof))))
    records=[]
    a=oof['anchor'].prediction
    for name in allowed:
        if name not in candidates:
            continue
        item=oof[name]
        score=metrics(item.prediction,fit_labels,item.probability)
        rh=rescue_harm(item.prediction,a,fit_labels)
        eligible=True
        if name!='anchor' and rh['net']<cfg['selection']['min_fit_oof_gain_count']:
            eligible=False
        limit=cfg['selection']['max_fit_oof_harm_count']
        if limit is not None and rh['harm']>limit:
            eligible=False
        records.append({'recipe':name,'accuracy':score['accuracy'],'nll':score.get('nll'),**rh,'eligible':eligible})
    available=[r for r in records if r['eligible']]
    if not available:
        name=fallback
    else:
        # 정확도 우선, 동점은 harm/변경 수를 줄인다. holdout으로 선택하지 않는다.
        row=max(available,key=lambda r:(r['accuracy'],-r['harm'],-r['changed']))
        name=row['recipe']
    if name not in candidates:
        raise ValueError('선택한 레시피의 전체 예측이 없습니다.')
    return name,{'policy':'fit_oof','selected':name,'holdout_used_for_selection':False,
                 'fit_oof_tuning_diagnostics':records,'selection_scores_are_not_independent_final_scores':True}

def evaluation_table(candidates,indices,y,anchor,evaluation_role,cfg):
    rows=[]
    for name,item in candidates.items():
        prob=None if item.probability is None else item.probability[indices]
        score=metrics(item.prediction[indices],y[indices],prob,cfg['calibration']['ece_bins'])
        class_info=score.pop('by_class')
        rows.append({'recipe':name,'evaluation_role':evaluation_role,**score,
                     **rescue_harm(item.prediction[indices],anchor[indices],y[indices]),
                     'has_actual_or_meta_probability':prob is not None,'probability_semantics':item.semantics,
                     'per_class_json':json.dumps(class_info,ensure_ascii=False)})
    return pd.DataFrame(rows)


## 15. 결과 비교·오답 검토

전체 validation 진단·meta-holdout·fit OOF를 구분한다. 두 모델의 오류 보완, 4/8 모델 만장일치, core3, 투표 패턴을 실제 예측으로 집계한다. test 검토 목록에는 정답 추정이나 Public 역추론을 넣지 않는다.


In [ ]:
def prediction_frame(ids,item,labels=None,roles=None):
    frame=pd.DataFrame({'id':ids,'pred':[CLASSES[i] for i in item.prediction]})
    if labels is not None:
        frame['answer']=[CLASSES[i] for i in labels]
        frame['correct']=item.prediction==labels
    if roles is not None:
        frame['evaluation_role']=roles
    if item.probability is not None:
        for i,c in enumerate(CLASSES):
            frame[c]=item.probability[:,i]
        conf,margin=confidence_margin(item.probability)
        frame['confidence']=conf;frame['margin']=margin
    if item.counts is not None:
        for i,c in enumerate(CLASSES):
            frame['vote_score_'+c]=item.counts[:,i]
    return frame

def consensus_patterns(models,split,ids,labels=None):
    votes=np.stack([getattr(m,split+'_pred') for m in models])
    counts=vote_counts(votes)
    patterns=[':'.join(str(int(x)) for x in sorted(row[row>0],reverse=True)) for row in counts]
    frame=pd.DataFrame({'id':ids,'vote_pattern':patterns,'all_unanimous':counts.max(1)==len(models),
                        'max_votes':counts.max(1).astype(int)})
    for m in models:
        frame[m.name]=[CLASSES[i] for i in getattr(m,split+'_pred')]
    if labels is not None:
        frame['answer']=[CLASSES[i] for i in labels]
        first=votes[0]
        frame['unanimous_correct']=np.where(frame.all_unanimous,first==labels,None)
    return frame

def review_queue(models,test_ids,selected,anchor_name,cfg):
    patterns=consensus_patterns(models,'test',test_ids)
    anchor=next(m for m in models if m.name==anchor_name)
    patterns['selected']=[CLASSES[i] for i in selected.prediction]
    patterns['anchor_answer']=[CLASSES[i] for i in anchor.test_pred]
    patterns['changed_vs_anchor']=selected.prediction!=anchor.test_pred
    if selected.probability is not None:
        conf,margin=confidence_margin(selected.probability)
        patterns['selected_confidence']=conf;patterns['selected_margin']=margin
    queue=patterns.loc[~patterns.all_unanimous|patterns.changed_vs_anchor].copy()
    queue['review_status']='pending_human_review'
    queue['ground_truth']='unknown'
    limit=cfg['manual_review']['queue_limit']
    return queue if limit is None else queue.head(limit)


## 16. 실제 차트·실행 감사

추가 그래프 패키지 없이 실제 결과 HTML/SVG 차트를 저장한다. NPY 해시, 경로, class order, 분할 ID, 설정, 라이브러리 버전, soft/hard 출처를 남긴다.


In [ ]:
def write_report(folder,table,summary):
    records=table.to_dict('records')
    bars=[]
    for role in dict.fromkeys(r['evaluation_role'] for r in records):
        sub=[r for r in records if r['evaluation_role']==role]
        height=42+34*len(sub)
        shapes=[]
        for i,row in enumerate(sub):
            y=32+i*34;width=float(row['accuracy'])*600
            label=html.escape(row['recipe'])
            title=html.escape(f"{label}: {row['accuracy']:.6f} ({row['correct']}/{row['n']})")
            color='#285e85' if row['recipe']==summary['selected_recipe'] else '#78a5b8'
            shapes.append(f'<text x="10" y="{y+16}" font-size="12">{label}</text><rect x="170" y="{y}" width="{width:.2f}" height="22" rx="4" fill="{color}"><title>{title}</title></rect><text x="{180+width:.2f}" y="{y+16}" font-size="12">{row["accuracy"]:.4f}</text>')
        bars.append('<section><h2>'+html.escape(role)+'</h2><svg viewBox="0 0 860 '+str(height)+'" role="img" aria-label="실제 정확도 비교">'+''.join(shapes)+'</svg></section>')
    body=table.drop(columns=['per_class_json'],errors='ignore').to_html(index=False,escape=True,float_format=lambda x:f'{x:.6f}',classes='results',border=0)
    doc='''<!doctype html><html lang="ko"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>VQA Ensemble Actual Results</title><style>body{font:15px/1.6 system-ui,sans-serif;background:#f4f7fa;color:#183044;margin:0}main{max-width:1140px;margin:auto;padding:32px}h1{font-size:28px}section,.card{background:white;padding:20px;margin:18px 0;border-radius:12px;box-shadow:0 2px 10px #1830440c}svg{width:100%;height:auto}table{border-collapse:collapse;font-size:12px}th,td{border-bottom:1px solid #dde5ed;padding:8px;text-align:left;white-space:nowrap}th{position:sticky;top:0;background:#eaf1f6;cursor:pointer}#table{overflow:auto;max-height:620px}input{padding:10px;width:280px;border:1px solid #b9cad7;border-radius:6px}.muted{color:#52697c}</style><main><h1>VQA 앙상블 실제 결과</h1>'''
    doc+='<div class="card"><strong>선택: '+html.escape(summary['selected_recipe'])+'</strong><p class="muted">meta_fit은 학습 진단, fit_oof는 선택용 교차검증, meta_holdout은 선택 시 제외한 평가입니다. 원 VLM 선택에 이미 validation을 사용했다면 프로젝트 전체의 독립 평가가 아닙니다. Public/Private 성적은 여기서 측정하지 않았습니다.</p></div>'
    doc+=''.join(bars)+'<section><h2>실행 결과 비교</h2><input id="filter" placeholder="recipe 또는 평가 범위 검색"><div id="table">'+body+'</div></section>'
    doc+='''<script>document.getElementById('filter').addEventListener('input',e=>{let q=e.target.value.toLowerCase();document.querySelectorAll('tbody tr').forEach(r=>r.style.display=r.innerText.toLowerCase().includes(q)?'':'none')});document.querySelectorAll('th').forEach((h,i)=>h.addEventListener('click',()=>{const t=h.closest('table'),b=t.tBodies[0],rows=[...b.rows];let d=h.dataset.d==='1'?-1:1;h.dataset.d=String(d);rows.sort((a,z)=>{let x=a.cells[i].innerText,y=z.cells[i].innerText;return d*(x!==''&&y!==''&&isFinite(x)&&isFinite(y)?Number(x)-Number(y):x.localeCompare(y))});rows.forEach(r=>b.appendChild(r))}));</script></main></html>'''
    path=folder/'charts'/'report.html';path.parent.mkdir(exist_ok=True)
    path.write_text(doc,encoding='utf-8')

def write_run_readme(folder,summary):
    text=f'''# VQA ensemble run

- version: {VERSION}
- selected recipe: {summary['selected_recipe']}
- anchor: {summary['anchor']}
- meta-fit: {summary['n_meta_fit']}; meta-holdout: {summary['n_meta_holdout']}
- measured meta-holdout accuracy: {summary['selected_holdout']['accuracy']:.6f}
- validation full: {summary['n_valid']} (mixed fit diagnostics + holdout; not an independent score)
- validation identity: {summary['validation_signature']}
- public/private: unmeasured

`config.json`, `environment.json`, `inputs.json`, `meta_split.json`, `fit_parameters.json`,
`analysis/selection.json`, `analysis/comparison.csv` describe the exact frozen run.

## Probability semantics

Only actual probability or trained meta-model probability outputs are saved as `*_probs.npy`.
Hard answers/vote counts are separate. `*_logits.npy` is not fabricated.
Full validation arrays include meta-fit in-sample outputs; evaluate the holdout separately.
Optional `meta_fit_oof_*` files are the meta-stage cross-fit outputs, not fresh VLM folds.

## Submission

Only export mode with the exact sample submission ID set generates final test CSVs.
All candidate submissions are comparisons, not leaderboard-measured results.
test-only recipes and manual repairs are separately named and unvalidated.

## Limits

meta-holdout independence only concerns this ensemble step. Base-model selection may have
already reused this validation. Images/near duplicates not present here cannot be audited
fully. Do not infer an individual test answer from an aggregate Public score or tune
repeatedly to a hidden result. This run does not recreate historical RF_v1/A changes.
'''
    (folder/'README.md').write_text(text,encoding='utf-8')


## 17. 제출 CSV·NPY 저장

sample_submission의 정확한 ID 순서로 출력한다. hard-only 결과는 `*_predictions.npy`·`*_vote_counts.npy`로 저장하고 `*_probs.npy`를 만들지 않는다. 앙상블에는 물리적 raw logits가 없어 가짜 logits 파일을 만들지 않는다.


In [ ]:
def export_candidate(folder,item,split,ids,labels=None,prefix=None,meta_in_sample=False):
    stem=split if prefix is None else prefix+'_'+split
    probs=folder/'probs'
    save_json(probs/f'{stem}_ids.json',ids)
    save_npy(probs/f'{stem}_predictions.npy',item.prediction.astype(np.int64))
    if labels is not None:
        save_npy(probs/f'{stem}_labels.npy',labels.astype(np.int64))
    if item.probability is not None:
        p=np.asarray(item.probability)
        if p.shape!=(len(ids),4) or not np.isfinite(p).all() or not np.allclose(p.sum(1),1,atol=1e-7):
            raise AssertionError('저장할 후보 확률 오류')
        if not np.array_equal(p.argmax(1),item.prediction):
            raise AssertionError('확률 argmax와 최종 답이 다릅니다.')
        save_npy(probs/f'{stem}_probs.npy',p.astype(np.float32))
    if item.counts is not None:
        save_npy(probs/f'{stem}_vote_counts.npy',item.counts)
    return {'shape':[len(ids),4] if item.probability is not None else None,
            'class_order':CLASSES,'original_option_order':True,'hard_onehot_used':False,
            'is_smoke':False,'meta_in_sample_present':meta_in_sample,
            'probs_semantics':item.semantics if item.probability is not None else 'not available; hard answer/counts only',
            'kind':'probability' if item.probability is not None else 'hard_answer_only',
            'raw_logits_available':False,'vote_counts_are_soft':False,'recipe':item.name}

def make_submission(folder,name,ids,pred):
    if len(ids)!=len(pred) or len(set(ids))!=len(ids) or not np.isin(pred,[0,1,2,3]).all():
        raise ValueError('제출 ID/행 수/정답 오류')
    frame=pd.DataFrame({'id':ids,'answer':[CLASSES[i] for i in pred]})
    path=folder/f'submission_{name}.csv'
    save_csv(path,frame)
    loaded=read_table(path)
    if list(loaded.columns)!=['id','answer'] or loaded.id.tolist()!=ids or not loaded.answer.isin(CLASSES).all():
        raise AssertionError('제출 CSV 저장 후 검증 실패')
    return path


## 18. Test-only·수동 검토 선택 분기

validation 없는 모델은 학습·후보 선택에서 제외한다. 명시한 고정 test-only 결합은 검증 성적 없이 별도 이름으로만 저장한다. 수동 수정은 기본 꺼짐이며 before/after/근거 기록을 요구하고 별도 파일에 저장한다.


In [ ]:
def run_test_only_recipes(models,ids,cfg,folder):
    by_name={m.name:m for m in models};records=[]
    for recipe in cfg['test_only_recipes']:
        name=recipe['name']
        if not name or any(c in name for c in '/\\\n\r'):
            raise ValueError('test-only recipe 이름 오류')
        names=recipe['models']
        if len(set(names))!=len(names) or not names or set(names)-set(by_name):
            raise ValueError('test-only recipe 모델 이름 오류')
        if not any(by_name[n].role=='test_only' for n in names):
            raise ValueError('test_only_recipes는 validation 미보유 모델을 명시하는 별도 경로입니다.')
        anchor_name=recipe.get('anchor',names[0])
        if anchor_name not in names:
            raise ValueError('test-only tie anchor는 등록한 recipe 모델이어야 합니다.')
        w=fixed_weights(names,recipe.get('weights',{}))
        if recipe['method']=='soft_mean':
            if any(by_name[n].test_probs is None for n in names):
                raise ValueError('test-only soft mean에 hard 모델을 넣을 수 없습니다.')
            q=np.tensordot(w,np.stack([by_name[n].test_probs for n in names]),axes=(0,0))
            item=Candidate(name,q.argmax(1),q,semantics='fixed test-only real soft mean; validation unmeasured')
        elif recipe['method']=='hard_vote':
            votes=np.stack([by_name[n].test_pred for n in names]);counts=vote_counts(votes,w)
            item=Candidate(name,resolve_tie(counts,by_name[anchor_name].test_pred),counts=counts,
                           semantics='fixed test-only hard weighted vote scores; validation unmeasured')
        else:
            raise ValueError('test-only method는 soft_mean / hard_vote입니다.')
        target=folder/'test_only'/name
        target.mkdir(parents=True)
        meta=export_candidate(target,item,'test',ids)
        save_json(target/'probs'/'probability_metadata.json',{'test':meta,'validation_available':False})
        path=make_submission(target,'TESTONLY_'+name,ids,item.prediction)
        records.append({'name':name,'submission':str(path),'validation_measured':False,
                        'selected_by_validation':False,'public_private':None,'recipe':recipe})
    return records

def manual_changes(folder,submission,cfg):
    options=cfg['manual_review']
    if not options['enabled']:
        return None
    if not options['changes_csv']:
        raise ValueError('수동 변경 활성화 시 changes_csv가 필요합니다.')
    changes=read_table(options['changes_csv'])
    needed=['id','before','after','reason']
    if set(needed)-set(changes):
        raise ValueError('수동 수정 CSV에는 id,before,after,reason 열이 필요합니다.')
    check_ids(changes.id.tolist(),'manual changes')
    source=read_table(submission).set_index('id',drop=False)
    if set(changes.id)-set(source.index):
        raise ValueError('수동 수정에 test 외 ID가 있습니다.')
    audit=[]
    for row in changes.to_dict('records'):
        if row['before'] not in CLASSES or row['after'] not in CLASSES or row['before']==row['after']:
            raise ValueError('수동 before/after는 다른 a/b/c/d여야 합니다.')
        if source.loc[row['id'],'answer']!=row['before']:
            raise ValueError('수동 before가 선택한 제출 답과 다릅니다.')
        if options['require_reason'] and not row['reason'].strip():
            raise ValueError('수동 변경 근거를 남기세요.')
        source.loc[row['id'],'answer']=row['after'];audit.append(row)
    path=folder/'submission_MANUAL_reviewed.csv'
    save_csv(path,source.reset_index(drop=True))
    save_csv(folder/'analysis'/'manual_changes_applied.csv',pd.DataFrame(audit))
    save_json(folder/'analysis'/'manual_changes_summary.json',{'n_changed':len(audit),'source':str(submission),
              'changes_sha256':hash_file(options['changes_csv']),'submission':str(path),
              'validation_measured':False,'public_private':None,'soft_probabilities_updated':False})
    return str(path)


## 19. 실행기·CLI·CPU 자체 검사

모든 단계가 실제 run에 연결된다. 단일 `.py --config ... --mode ...` 실행도 가능하다. CPU 자체 검사는 생성한 예제임을 표시하고 실험 성적으로 저장하지 않는다.


In [ ]:
def create_run_dir(cfg,mode):
    root=Path(cfg['output_root']).expanduser().resolve()
    name=cfg['run_id'] or ('ensemble_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')+'_'+mode)
    if not name or name in {'.','..'} or any(c in name for c in '/\\\n\r'):
        raise ValueError('run_id는 폴더 이름이어야 합니다.')
    folder=root/name
    if folder.exists():
        raise FileExistsError('기존 run은 덮어쓰지 않습니다. 새 run_id를 사용하세요: '+str(folder))
    folder.mkdir(parents=True)
    (folder/'analysis').mkdir();(folder/'probs').mkdir()
    return folder

def run_ensemble(config,mode='inspect'):
    cfg=deepcopy(config);validate_config(cfg)
    if mode not in {'inspect','evaluate','export'}:
        raise ValueError('run mode는 inspect / evaluate / export입니다.')
    if not cfg['models']:
        raise ValueError('완료된 모델 run을 CFG["models"]에 등록하세요.')
    need_test=mode=='export' or (mode=='inspect' and bool(cfg['sample_submission_csv']))
    models=[load_model(s,cfg,need_test) for s in cfg['models']]
    complete=[m for m in models if m.role=='complete']
    canonical=canonical_validation(models,cfg)
    test=canonical_test(models,cfg) if need_test else None
    folder=create_run_dir(cfg,mode)
    save_json(folder/'config.json',cfg);save_json(folder/'environment.json',environment())
    save_json(folder/'inputs.json',{'models':[{**m.audit,'name':m.name,'validation_signature':m.validation_signature,'spec':m.spec} for m in models],
             'validation_csv':cfg['validation_csv'],'validation_csv_sha256':hash_file(cfg['validation_csv']) if cfg['validation_csv'] else None,
             'canonical_validation_signature':canonical['signature'],'test':test,'warnings':canonical['warnings']})
    save_csv(folder/'analysis'/'input_catalog.csv',pd.DataFrame(input_catalog(models)))
    if mode=='inspect':
        info={'mode':mode,'run_dir':str(folder),'n_valid':len(canonical['ids']),
              'n_test':None if test is None else test['n'],'group_column':canonical['group_column'],
              'n_groups':len(np.unique(canonical['groups'])),'warnings':canonical['warnings'],
              'ensemble_fitted':False,'submission_created':False,'models':input_catalog(models)}
        save_json(folder/'execution_status.json',info)
        return info
    y=canonical['labels'];ids=canonical['ids'];n=len(ids)
    fit,hold,split_info=make_meta_split(canonical,cfg)
    save_json(folder/'meta_split.json',split_info)
    anchor=choose_anchor(complete,fit,y,cfg);ranking=rank_models(complete,fit,y)
    state=fit_state(complete,fit,y,cfg,anchor,ranking)
    valid_candidates=recipe_candidates(complete,'valid',np.arange(n),state,cfg)
    need_oof=(cfg['selection']['policy']=='fit_oof' or cfg['selection']['force_inner_oof']
              or cfg['calibration']['mode']!='none' or cfg['learned_weights']['enabled']
              or bool(state['estimators']))
    oof={};oof_audit={}
    if need_oof:
        oof,oof_audit=build_fit_oof(complete,fit,canonical,cfg)
        save_json(folder/'analysis'/'meta_fit_oof_audit.json',oof_audit)
    selected,selection=select_recipe(valid_candidates,oof,y[fit],cfg)
    # 선택을 저장한 뒤 holdout 점수를 계산한다. 선택 함수에 holdout 정답을 주지 않는다.
    selection['config_signature']=digest(cfg);selection['meta_split_signature']=split_info['signature']
    save_json(folder/'analysis'/'selection.json',selection)
    save_json(folder/'fit_parameters.json',state_summary(state,ids))
    anchor_prediction=next(m for m in complete if m.name==anchor).valid_pred
    tables=[evaluation_table(valid_candidates,fit,y,anchor_prediction,'meta_fit_in_sample_diagnostic',cfg),
            evaluation_table(valid_candidates,hold,y,anchor_prediction,'meta_holdout',cfg),
            evaluation_table(valid_candidates,np.arange(n),y,anchor_prediction,'full_validation_mixed_diagnostic',cfg)]
    if oof:
        tables.append(evaluation_table(oof,np.arange(len(fit)),y[fit],oof['anchor'].prediction,'meta_fit_oof_selection_diagnostic',cfg))
    comparison=pd.concat(tables,ignore_index=True)
    save_csv(folder/'analysis'/'comparison.csv',comparison)
    save_csv(folder/'analysis'/'pairwise_full_validation_diagnostic.csv',pairwise_comparison(complete,y))
    save_csv(folder/'analysis'/'validation_consensus_patterns.csv',consensus_patterns(complete,'valid',ids,y))
    base_rows=[]
    for m in complete:
        score=metrics(m.valid_pred,y,m.valid_probs,cfg['calibration']['ece_bins']);score.pop('by_class')
        base_rows.append({'model':m.name,'scope':'common_base_model_validation',**score})
    save_csv(folder/'analysis'/'base_model_validation.csv',pd.DataFrame(base_rows))
    roles=np.full(n,'meta_fit_in_sample_diagnostic',dtype=object);roles[hold]='meta_holdout'
    item=valid_candidates[selected]
    save_csv(folder/'validation_predictions.csv',prediction_frame(ids,item,y,roles))
    metadata={'valid':export_candidate(folder,item,'valid',ids,y,meta_in_sample=bool(need_oof or cfg['anchor']['name'] is None)),
              'class_order':CLASSES,'producer':'VQA_Ensemble_Baseline','raw_logits_fabricated':False}
    save_csv(folder/'analysis'/'selected_holdout_predictions.csv',prediction_frame([ids[i] for i in hold],
          Candidate(selected,item.prediction[hold],None if item.probability is None else item.probability[hold],
                    None if item.counts is None else item.counts[hold],item.semantics),y[hold]))
    if selected in oof:
        metadata['meta_fit_oof_valid']=export_candidate(folder,oof[selected],'valid',[ids[i] for i in fit],y[fit],prefix='meta_fit_oof')
        save_csv(folder/'analysis'/'selected_fit_oof_predictions.csv',prediction_frame([ids[i] for i in fit],oof[selected],y[fit]))
    hold_prob=None if item.probability is None else item.probability[hold]
    heldout=metrics(item.prediction[hold],y[hold],hold_prob,cfg['calibration']['ece_bins'])
    heldout.update(rescue_harm(item.prediction[hold],anchor_prediction[hold],y[hold]))
    summary={'schema':'ensemble_validation_summary_v1','mode':mode,'selected_recipe':selected,'anchor':anchor,
             'n_valid':n,'n_meta_fit':len(fit),'n_meta_holdout':len(hold),'n_test':None,
             'selected_holdout':heldout,'validation_signature':canonical['signature'],
             'base_validation_signature':canonical['base_validation_signature'],
             'accuracy':float((item.prediction==y).mean()),'accuracy_semantics':'full validation mixed diagnostic, not independent evaluation',
             'is_smoke':False,'public_private_measured':False,'holdout_used_for_selection':False,
             'warnings':canonical['warnings'],'selected_probability_semantics':item.semantics,
             'base_vlm_selection_independence_unverified':True,'training_gpu_used':False,
             'run_dir':str(folder),'submission':None}
    if mode=='export':
        test_ids=test['ids'];nt=len(test_ids)
        test_candidates=recipe_candidates(complete,'test',np.arange(nt),state,cfg)
        if set(valid_candidates)!=set(test_candidates):
            raise AssertionError('validation/test 후보 구성이 다릅니다.')
        chosen=test_candidates[selected]
        metadata['test']=export_candidate(folder,chosen,'test',test_ids)
        save_csv(folder/'test_predictions.csv',prediction_frame(test_ids,chosen))
        path=make_submission(folder,'selected_'+selected,test_ids,chosen.prediction)
        summary.update(n_test=nt,submission=str(path))
        if cfg['artifacts']['save_all_candidate_submissions']:
            for name,candidate in test_candidates.items():
                target=folder/'candidates'/name;target.mkdir(parents=True)
                make_submission(target,name,test_ids,candidate.prediction)
                candidate_meta=export_candidate(target,candidate,'test',test_ids)
                save_json(target/'probs'/'probability_metadata.json',{'test':candidate_meta,'public_private_measured':False})
        if cfg['manual_review']['export_queue']:
            save_csv(folder/'analysis'/'test_review_queue.csv',review_queue(complete,test_ids,chosen,anchor,cfg))
        test_only=run_test_only_recipes(models,test_ids,cfg,folder)
        save_json(folder/'analysis'/'test_only_recipes.json',test_only)
        summary['manual_submission']=manual_changes(folder,path,cfg)
    elif cfg['test_only_recipes'] or cfg['manual_review']['enabled']:
        summary['warnings'].append('test-only/manual 경로는 export에서만 실행합니다.')
    save_json(folder/'probs'/'probability_metadata.json',metadata)
    save_json(folder/'validation_summary.json',summary)
    save_json(folder/'execution_status.json',{'mode':mode,'meta_fitted':True,'gpu_training':False,
              'selected_recipe':selected,'validation_evaluated':True,'test_ensembled':mode=='export',
              'public_private_measured':False,'submission':summary['submission']})
    if cfg['artifacts']['save_aligned_inputs']:
        for m in complete:
            save_npy(folder/'aligned_inputs'/f'{m.name}_valid_hard.npy',m.valid_pred)
            if m.valid_probs is not None:
                save_npy(folder/'aligned_inputs'/f'{m.name}_valid_real_probs.npy',m.valid_probs.astype(np.float32))
    if cfg['artifacts']['write_charts']:
        write_report(folder,comparison,summary)
    write_run_readme(folder,summary)
    return summary

def self_check():
    """CPU 합성 fixture 검사. 실제 대회 데이터/점수 검증이 아니다."""
    cfg=default_config()
    p=np.array([[.1,.2,.6,.1],[.65,.1,.15,.1]])
    assert np.array_equal(align_array(p,['002','001'],['001','002'],'fixture'),p[[1,0]])
    assert class_permutation(['d','c','b','a'])==[3,2,1,0]
    for bad in ([['01','01'],['01','02']],):
        try:
            check_ids(bad[0],'fixture duplicate')
            raise AssertionError('ID 중복을 놓쳤습니다.')
        except ValueError:
            pass
    try:
        check_probs(np.eye(4),4,'fixture onehot',cfg)
        raise AssertionError('hard onehot을 놓쳤습니다.')
    except ValueError:
        pass
    q=temperature_probs(p,1.7)
    assert np.allclose(q.sum(1),1) and np.array_equal(q.argmax(1),p.argmax(1))
    votes=np.array([[0,1],[1,0]])
    assert np.array_equal(resolve_tie(vote_counts(votes),np.array([1,2])),[1,0])
    labels=np.tile(np.arange(4),30)
    groups=np.repeat(np.arange(60),2)
    folds=grouped_folds(labels,groups,5,42)
    seen=np.zeros(len(labels),int)
    for tr,va in folds:
        assert not set(groups[tr])&set(groups[va]);seen[va]+=1
    assert np.all(seen==1)
    with tempfile.TemporaryDirectory() as directory:
        folder=Path(directory)
        item=Candidate('fixture_hard',np.array([0,1]),counts=np.array([[1,0,0,0],[0,1,0,0]]))
        export_candidate(folder,item,'test',['001','002'])
        assert not (folder/'probs'/'test_probs.npy').exists()
        make_submission(folder,'FIXTURE',['001','002'],item.prediction)
        assert read_table(folder/'submission_FIXTURE.csv').id.tolist()==['001','002']
    return {'status':'passed','data':'CPU synthetic fixtures only','actual_competition_score':None,
            'checks':['ID preservation/alignment','class remapping','onehot rejection','group folds','temperature','tie','hard-only export']}

def main():
    parser=argparse.ArgumentParser(description='Reusable VQA ensemble on stored probabilities/hard answers')
    parser.add_argument('--config',type=Path)
    parser.add_argument('--mode',choices=['inspect','evaluate','export','self_check'],default='inspect')
    parser.add_argument('--write-default-config',type=Path)
    args=parser.parse_args()
    if args.write_default_config:
        save_json(args.write_default_config,default_config());return
    if args.mode=='self_check':
        print(json.dumps(self_check(),ensure_ascii=False,indent=2));return
    if not args.config:
        parser.error('실제 run에는 --config가 필요합니다.')
    result=run_ensemble(read_json(args.config),args.mode)
    print(json.dumps(jsonable(result),ensure_ascii=False,indent=2))


## 20. 실행 설정·실행

처음에는 `RUN_MODE="inspect"`로 입력을 검사한다. `evaluate`는 공통 validation만 비교하고, `export`는 동결한 규칙으로 test 제출 파일을 저장한다. `self_check`는 CPU 합성 fixture 검사이며 대회 성적이 아니다. 경로 등록 없이 실행하면 필요한 설정 안내만 표시한다.


In [ ]:
CFG = default_config()
AUTO_RUN = True
RUN_MODE = "inspect"  # inspect / evaluate / export / self_check

# 실제로 완료된 run 폴더를 등록한다. 같은 split manifest로 valid를 다시 추론해야 한다.
# CFG["models"] = [
#     {"name": "gemma31", "kind": "soft_run", "run_dir": "/content/drive/MyDrive/vqa/runs/gemma31",
#      "role": "complete", "heldout_predictions_confirmed": True},
#     {"name": "qwen27_ocr", "kind": "soft_run", "run_dir": "/content/drive/MyDrive/vqa/runs/qwen27_ocr",
#      "role": "complete", "heldout_predictions_confirmed": True},
#     {"name": "agnes", "kind": "soft_run", "run_dir": "/content/drive/MyDrive/vqa/runs/agnes",
#      "role": "complete", "heldout_predictions_confirmed": True},
# ]
# CFG["validation_csv"] = "/content/drive/MyDrive/vqa/runs/gemma31/validation_rows.csv"
# CFG["sample_submission_csv"] = "/content/drive/MyDrive/vqa/data/sample_submission.csv"
# CFG["output_root"] = "/content/drive/MyDrive/vqa/ensemble_runs"
# CFG["anchor"]["name"] = "gemma31"  # None이면 meta-fit 정확도로 선택
# CFG["calibration"]["mode"] = "final_probs_temperature"  # 기본 none
# CFG["learned_weights"]["enabled"] = True
# CFG["meta_models"]["logistic"]["enabled"] = True
# CFG["meta_models"]["rf"]["enabled"] = True
# CFG["selection"]["policy"] = "fit_oof"  # 기본 fixed; 내부 OOF에서만 선택

# 저장된 JSON을 사용하려면:
# CFG = json.loads(Path("VQA_Ensemble_Config.json").read_text(encoding="utf-8"))

if AUTO_RUN:
    if RUN_MODE == "self_check":
        print(self_check())
    elif not CFG["models"]:
        print("20번 셀에서 모델 run 경로를 등록하세요. 아직 실제 앙상블·제출 파일은 생성하지 않았습니다.")
    else:
        RESULT = run_ensemble(CFG, RUN_MODE)
        print(json.dumps(RESULT, ensure_ascii=False, indent=2))


### 출처·한계

- 직접 연결한 입력 규격: `VQA_Gemma31B_Reusable_Baseline_v1.ipynb`의 05·21단계(`validation_rows.csv`, 실제 확률·IDs·정답·metadata).
- [NumPy load](https://numpy.org/doc/stable/reference/generated/numpy.load.html): 안전한 NPY 읽기.
- [scikit-learn stacking 안내](https://scikit-learn.org/stable/auto_examples/ensemble/plot_stack_predictors.html): OOF 예측을 메타 모델에 사용.
- [scikit-learn calibration 안내](https://scikit-learn.org/stable/modules/calibration.html): 분리된 데이터에서 보정 평가.
- [StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html): 그룹과 클래스 분포를 함께 고려.

meta-holdout은 앙상블 단계에서 제외한 데이터다. 원래 VLM checkpoint/프롬프트를 선택하는 데 validation을 반복 사용했다면 전체 프로젝트 수준의 완전한 독립 평가라고 부를 수 없다. 그룹 CSV를 확인해도 모든 원본 이미지 근접 중복을 자동 검증하는 것은 아니다.

사용자가 제공한 기존 최고 A(Public 0.98182 / Private 0.98302)와 새로운 베이스라인 실행은 별개다. A의 실제 RF 학습 파일·15개 변경 감사 기록을 입력하지 않았으므로 재현이나 최적해를 주장하지 않는다. Public 점수는 aggregate이며 개별 test 정답을 확정하는 근거가 아니다. 결과 확인 후 반복 수정·재제출하면 hidden/test 성능을 과적합할 수 있다.
